In [6]:
import geopandas as gpd
from shapely.geometry import Point

# 1. Load your exported file

gdf = gpd.read_file('./LWD/export (1).geojson')

# 2. Project to a local coordinate system (meters) for accurate area calculation
# UTM zone 46N is usually good for Dhaka
gdf = gdf.to_crs(epsg=32646)

# 3. Calculate total building footprint area
total_building_area = gdf.area.sum()

# 4. Calculate your search area (1km radius circle)
# Area = π * r^2
search_radius_m = 1000
total_search_area = 3.14159 * (search_radius_m**2)

# 5. Get the density percentage
density = (total_building_area / total_search_area) * 100

print(f"Building Density: {density:.2f}%")

Building Density: 42.52%


In [7]:
import geopandas as gpd

# 1. Load the specific file from your LWD folder (noticing the name from your tab)
gdf = gpd.read_file('./LWD/export (1).geojson')

# 2. Filter to keep only actual building polygons (removes points/lines)
buildings_gdf = gdf[gdf.geometry.type.isin(['Polygon', 'MultiPolygon'])]

# 3. Calculate Building Count
building_count = len(buildings_gdf)

# 4. Area Calculations (using UTM 46N for Dhaka)
buildings_projected = buildings_gdf.to_crs(epsg=32646)
total_building_area = buildings_projected.area.sum()

search_radius_m = 1000
total_search_area_m2 = 3.14159 * (search_radius_m**2)
total_search_area_km2 = total_search_area_m2 / 1_000_000

# 5. Final Metrics
density_pct = (total_building_area / total_search_area_m2) * 100
buildings_per_km2 = building_count / total_search_area_km2

print(f"Total Buildings Found: {building_count}")
print(f"Building Density: {density_pct:.2f}%")
print(f"Buildings per sq km: {buildings_per_km2:.2f}")

Total Buildings Found: 10021
Building Density: 42.52%
Buildings per sq km: 3189.79


In [8]:
import geopandas as gpd

# 1. Load the road data
# Make sure the filename matches what you saved from Overpass
roads_gdf = gpd.read_file('./LWD/export (1).geojson')

# 2. Filter out non-road features (optional but recommended)
# This keeps only actual street segments
roads_gdf = roads_gdf[roads_gdf.geometry.type.isin(['LineString', 'MultiLineString'])]

# 3. Project to UTM 46N (Dhaka's local coordinate system) 
# This converts degrees to meters so we can calculate length accurately
roads_projected = roads_gdf.to_crs(epsg=32646)

# 4. Calculate Total Road Length in Kilometers
total_length_meters = roads_projected.geometry.length.sum()
total_length_km = total_length_meters / 1000

# 5. Define Study Area (1km radius circle = 3.14159 sq km)
study_area_km2 = 3.14159

# 6. Calculate Road Density (km of road per sq km of land)
road_density = total_length_km / study_area_km2

print(f"Total Road Length: {total_length_km:.2f} km")
print(f"Road Density: {road_density:.2f} km/km²")

Total Road Length: 97.37 km
Road Density: 30.99 km/km²


In [9]:
import requests
import numpy as np

def get_avg_elevation(center_lat, center_lon, radius_m=1000, samples=5):
    # 1. Create a grid of points to sample across the area
    # (Mirpur is flat, so a small grid is sufficient)
    lat_offset = radius_m / 111000  # Approx meters to degrees
    lon_offset = radius_m / (111000 * np.cos(np.radians(center_lat)))
    
    lats = np.linspace(center_lat - lat_offset, center_lat + lat_offset, samples)
    lons = np.linspace(center_lon - lon_offset, center_lon + lat_offset, samples)
    
    all_elevations = []
    
    # 2. Call the Open-Meteo Elevation API
    # You can batch up to 100 locations in one request
    lat_str = ",".join(map(str, lats.flatten()))
    lon_str = ",".join(map(str, lons.flatten()))
    
    url = f"https://api.open-meteo.com/v1/elevation?latitude={lat_str}&longitude={lon_str}"
    
    try:
        response = requests.get(url)
        data = response.json()
        all_elevations = data.get('elevation', [])
        
        avg_elevation = sum(all_elevations) / len(all_elevations)
        return round(avg_elevation, 2)
    except Exception as e:
        print(f"Error fetching elevation: {e}")
        return None

# Mirpur 1 Center Coordinates
mirpur_lat, mirpur_lon = 23.7945, 90.3512
avg_el = get_avg_elevation(mirpur_lat, mirpur_lon)

print(f"Average Elevation for Mirpur-1 area: {avg_el} meters")

Average Elevation for Mirpur-1 area: 13.4 meters


In [11]:
import osmnx as ox
import geopandas as gpd
import requests
import pandas as pd
from shapely.geometry import Point

# Areas list stays the same
areas = [
    {"name": "Mirpur-1", "lat": 23.7945, "lon": 90.3512},
    {"name": "Mirpur-2", "lat": 23.8055, "lon": 90.3630},
    {"name": "Mirpur-6", "lat": 23.8123, "lon": 90.3598},
    {"name": "Mirpur-7", "lat": 23.8180, "lon": 90.3640},
    {"name": "Mirpur-9", "lat": 23.8010, "lon": 90.3580},
    {"name": "Mirpur-10", "lat": 23.8069, "lon": 90.3687},
    {"name": "Mirpur-11", "lat": 23.8140, "lon": 90.3725},
    {"name": "Mirpur-12", "lat": 23.8245, "lon": 90.3655},
    {"name": "Mirpur-13", "lat": 23.8090, "lon": 90.3780},
    {"name": "Mirpur-14", "lat": 23.8005, "lon": 90.3850},
    {"name": "Kazipara", "lat": 23.7930, "lon": 90.3735},
    {"name": "Sheorapara", "lat": 23.7875, "lon": 90.3750},
    {"name": "Pallabi", "lat": 23.8210, "lon": 90.3620},
    {"name": "Senpara Parbata", "lat": 23.8035, "lon": 90.3720},
    {"name": "Rupnagar", "lat": 23.8150, "lon": 90.3530},
    {"name": "Bhashantek", "lat": 23.8155, "lon": 90.3880},
    {"name": "Darus Salam", "lat": 23.7870, "lon": 90.3510},
    {"name": "Shah Ali", "lat": 23.7980, "lon": 90.3450},
    {"name": "Shialbari", "lat": 23.8085, "lon": 90.3515}
]

def get_urban_metrics(loc, index):
    lat, lon = loc['lat'], loc['lon']
    dist = 1000  
    area_km2 = 3.14159  
    
    print(f"Processing: {loc['name']}...")
    
    try:
        # 1. BUILDINGS
        buildings = ox.features_from_point((lat, lon), tags={'building': True}, dist=dist)
        buildings = buildings[buildings.geometry.type.isin(['Polygon', 'MultiPolygon'])]
        buildings_proj = buildings.to_crs(epsg=32646)
        b_area = buildings_proj.area.sum() / 1_000_000 
        b_density = (b_area / area_km2) * 100
        
        # 2. ROADS & INTERSECTIONS
        G = ox.graph_from_point((lat, lon), dist=dist, network_type='all')
        stats = ox.basic_stats(G)
        r_density = (stats['street_length_total'] / 1000) / area_km2
        i_density = stats['intersection_count'] / area_km2
        
        # 3. ELEVATION
        el_url = f"https://api.open-meteo.com/v1/elevation?latitude={lat}&longitude={lon}"
        elevation = requests.get(el_url).json()['elevation'][0]
        
        # 4. NEAREST WATER (Fixed Geometry logic)
        water = ox.features_from_point((lat, lon), tags={'natural': 'water', 'waterway': True}, dist=5000)
        water_proj = water.to_crs(epsg=32646)
        # Explicitly create a Point geometry for the center
        center_geom = Point(lon, lat) 
        center_gs = gpd.GeoSeries([center_geom], crs="EPSG:4326").to_crs(epsg=32646)
        dist_to_water = water_proj.distance(center_gs[0]).min() / 1000 

    except Exception as e:
        print(f"Error in {loc['name']}: {e}")
        return None

    return {
        "grid_id": index + 1,
        "area_name": loc['name'],
        "building_area_km_2": round(b_area, 4),
        "building_density": round(b_density, 2),
        "road_density_km_2": round(r_density, 2),
        "intersection_density": round(i_density, 2),
        "avg_elevation": elevation,
        "nearest_water": round(dist_to_water, 3),
        "pop_density": "", 
        "risk_level": ""    
    }

# Execution
final_results = []
for i, loc in enumerate(areas):
    data = get_urban_metrics(loc, i)
    if data:
        final_results.append(data)

df = pd.DataFrame(final_results)
df.to_csv('./LWD/Original_Dataset.csv', index=False)
print(f"\nSaved {len(df)} areas to './LWD/Original_Dataset.csv'")

Processing: Mirpur-1...
Processing: Mirpur-2...
Processing: Mirpur-6...
Processing: Mirpur-7...
Processing: Mirpur-9...
Processing: Mirpur-10...
Processing: Mirpur-11...
Processing: Mirpur-12...
Processing: Mirpur-13...
Processing: Mirpur-14...
Processing: Kazipara...
Processing: Sheorapara...
Processing: Pallabi...
Processing: Senpara Parbata...
Processing: Rupnagar...
Processing: Bhashantek...
Processing: Darus Salam...
Processing: Shah Ali...
Processing: Shialbari...

Saved 19 areas to './LWD/Original_Dataset.csv'


In [12]:
import pandas as pd

# 1. Load your existing dataset
df = pd.read_csv('./LWD/Original_Dataset.csv')

# 2. Define the total study area (1km radius circle in km2)
TOTAL_AREA_KM2 = 3.14159

# 3. Calculate the percentage in a 0-1 range
# This takes the physical area and divides it by the total possible area
df['building_density'] = df['building_area_km_2'] / TOTAL_AREA_KM2

# 4. Optional: Round to 4 decimal places for cleanliness
df['building_density'] = df['building_density'].round(4)

# 5. Save the updated dataset back to the CSV
df.to_csv('./LWD/Original_Dataset.csv', index=False)

print("Column 'building_density' has been reset to a 0-1 scale.")
print(df[['area_name', 'building_area_km_2', 'building_density']].head())

Column 'building_density' has been reset to a 0-1 scale.
  area_name  building_area_km_2  building_density
0  Mirpur-1              1.6155            0.5142
1  Mirpur-2              1.8582            0.5915
2  Mirpur-6              1.7721            0.5641
3  Mirpur-7              1.9789            0.6299
4  Mirpur-9              1.8469            0.5879
